# Cell identity transferred onto human nuclei, on the joint SAMap UMAP

The twin of Fig. 6C for the label transfer. Each human nucleus is colored by the mouse cell type (eFib or dec) that
carries most of its cross-species edge weight in the SAMap graph: for human nucleus *i*, the vote for type *t* is the
sum of the connectivities from *i* to mouse nuclei of type *t*, and the label is the type with the larger vote (the
operator of `workflow/scripts/samap/samap_label_stain_full.py`). Every mouse nucleus votes, the apoptotic decidual ones
included, since they carry a cell type even though they have no pseudotime. Mouse nuclei sit underneath in dark grey
for context; a second version leaves them out. Human nuclei with no mouse neighbor are drawn larger in grey behind the
colored ones; they are not imputed.

Panel size, square frame, dot size, draw-order seed, axis arrows and grey context match the Fig. 6C notebook, so this
panel overlays 6A, 6B and 6C exactly. Colors and key labels are the manuscript's (Fig. 6B), from
`cross_species_panels.py`, beside this notebook. Every nucleus is drawn.

**Inputs**: `joint.parquet` (species and joint UMAP), the human x mouse block of the SAMap graph
(`results/samap_figures/graph_block`, rule `samap_graph_block`), and `mouse_cells.csv` / `human_cells.csv` (release
cell types; rule `samap_figure_tables`).
**Outputs**, in `OUT`: `umap_dark_transferred_cell_type.{png,svg,pdf}` and
`umap_dark_transferred_cell_type_humanonly.{png,svg,pdf}` (90 x 90 mm, 600 dpi), `transferred_cell_type.csv` (one row
per human nucleus: native and transferred cell type, vote share), and
`fig6_umap_transferred_cell_type.provenance.json` with the counts drawn and the agreement with the native labels.

**Running it.** The pipeline runs it with papermill (rule `samap_fig6_cell_type`, `workflow/rules/29_fig_samap.smk`).
By hand, from the repository root:

```bash
papermill notebooks/samap/fig6_umap_transferred_cell_type.ipynb \
    results/notebooks/samap/fig6_umap_transferred_cell_type.executed.ipynb --cwd notebooks/samap -p FORCE True
```

Commit this notebook without outputs: the provenance record notes whether the notebook matches its commit.

In [ ]:
# Parameters. papermill -p NAME VALUE overrides them in a cell below this one.
# Paths are relative to the repository root; REPO = None finds the root with git from the working directory.
REPO = None
JOINT = "results/samap_figures/tables/joint.parquet"
MOUSE_CELLS = "results/samap_figures/tables/mouse_cells.csv"
HUMAN_CELLS = "results/samap_figures/tables/human_cells.csv"
GRAPH_DIR = "results/samap_figures/graph_block"
OUT = "results/figures/samap_main"
FORCE = False  # overwrite existing outputs
NOTEBOOK = "notebooks/samap/fig6_umap_transferred_cell_type.ipynb"  # this notebook, for the provenance record

In [ ]:
import datetime
import hashlib
import json
import logging
import platform
import subprocess
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from IPython.display import Image, display
from matplotlib import font_manager
from matplotlib.lines import Line2D

# This notebook's messages in the cell output; other libraries stay at WARNING.
logging.basicConfig(format="%(message)s", stream=sys.stdout, force=True)
LOG = logging.getLogger("fig6_cell_type")
LOG.setLevel(logging.INFO)

## Constants

In [ ]:
# Helvetica Neue is a system font on macOS that matplotlib does not find on its own; elsewhere the family list
# falls through to DejaVu Sans.
FONT_FAMILY = ["Helvetica Neue", "Helvetica", "DejaVu Sans"]
MACOS_HELVETICA_NEUE = Path("/System/Library/Fonts/HelveticaNeue.ttc")
TEXT = "white"

VARIANTS = {
    "with_mouse": "umap_dark_transferred_cell_type",
    "human_only": "umap_dark_transferred_cell_type_humanonly",
}
TITLE = "Cell identity transferred onto human cells"

MOUSE_CONTEXT = "#3a3a3a"  # mouse nuclei: their shape shows, their color does not compete (as in 6C)
NO_VALUE = "#9a9a9a"       # human nuclei with no mouse SAMap neighbor (as in 6C)
AXIS_ARROWS = "#bbbbbb"

PANEL_MM = 90
UMAP_DOT = 0.5             # pt^2
NO_VALUE_DOT_SCALE = 3     # larger, so the few nuclei without a label stay visible
DPI = 600
ORDER_SEED = 3             # seeded random draw order, so neither cell type always sits on top

## Figure

In [ ]:
def find_repo(repo) -> Path:
    if repo is not None:
        return Path(repo)
    git_root = ["git", "rev-parse", "--show-toplevel"]
    return Path(subprocess.run(git_root, capture_output=True, text=True, check=True).stdout.strip())


def set_font() -> str:
    """Register Helvetica Neue on macOS and set the family list; returns the family that will be used."""
    if MACOS_HELVETICA_NEUE.exists():
        font_manager.fontManager.addfont(str(MACOS_HELVETICA_NEUE))
    mpl.rcParams.update({"font.family": FONT_FAMILY, "svg.fonttype": "none", "pdf.fonttype": 42})
    return font_manager.FontProperties(family=FONT_FAMILY).get_name()


def manuscript_colors(repo: Path):
    """Cell-type colors and key labels of Fig. 6B, from the cross-species panel module (one source for both)."""
    # The copy beside this notebook (xmens_final workflow/scripts/figures/panels/cross_species/), imported from the
    # working directory, which is this notebook's folder.
    import cross_species_panels

    return cross_species_panels.CELL_TYPE_COLORS, cross_species_panels.CELL_TYPE_LABELS


def file_record(path: Path, role: str) -> dict:
    stat = path.stat()
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return {
        "path": str(path.resolve()),
        "role": role,
        "bytes": stat.st_size,
        "mtime": datetime.datetime.fromtimestamp(stat.st_mtime).isoformat(timespec="seconds"),
        "sha256": digest.hexdigest(),
    }


def git_revision(path: Path):
    """Commit of the checkout holding path, and whether path matches it; None outside git."""
    git = ["git", "-C", str(path.parent)]
    try:
        commit = subprocess.run(git + ["rev-parse", "HEAD"], capture_output=True, text=True, check=True)
        status = subprocess.run(git + ["status", "--porcelain", "--", path.name],
                                capture_output=True, text=True, check=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None
    return {"commit": commit.stdout.strip(), "notebook_matches_commit": status.stdout.strip() == ""}


def transfer_cell_type(graph_dir: Path, mouse_cell_type: pd.Series) -> pd.DataFrame:
    """Edge-weighted vote of each human nucleus's mouse neighbors (rows of C_hs_mm) over the mouse cell types.

    Returns one row per human nucleus: transferred cell type (NaN with no mouse neighbor), the share of the
    cross-species weight behind it, the total weight, and whether the two votes tie exactly.
    """
    block = sp.load_npz(graph_dir / "C_hs_mm.npz").tocsr()
    hs_names = (graph_dir / "hs_names.txt").read_text().split()
    mm_names = (graph_dir / "mm_names.txt").read_text().split()
    if block.shape != (len(hs_names), len(mm_names)):
        raise ValueError(f"graph block {block.shape} does not match its name lists")
    labels = mouse_cell_type.reindex(mm_names)
    if labels.isna().any():
        raise ValueError(f"{int(labels.isna().sum())} mouse nuclei of the graph have no cell type")

    types = np.array(sorted(labels.unique()))
    onehot = sp.csr_matrix((np.ones(len(labels)), (np.arange(len(labels)), np.searchsorted(types, labels.to_numpy()))),
                           shape=(len(labels), len(types)))
    votes = np.asarray((block @ onehot).todense())
    total = votes.sum(axis=1)
    mapped = total > 1e-9
    top = votes.max(axis=1)
    transferred = np.where(mapped, types[votes.argmax(axis=1)], None)
    return pd.DataFrame({
        "transferred_cell_type": pd.Series(transferred, index=hs_names, dtype="object"),
        "vote_share": np.where(mapped, top / np.where(mapped, total, 1), np.nan),
        "cross_species_weight": total,
        "tie": mapped & ((votes == top[:, None]).sum(axis=1) > 1),
    }, index=pd.Index(hs_names, name="cell"))


def square_limits(umap: pd.DataFrame) -> tuple:
    """Square frame around the full joint embedding (both species), 6% wider than its larger extent (as in 6C)."""
    x, y = umap["X_umap__0"], umap["X_umap__1"]
    side = max(x.max() - x.min(), y.max() - y.min()) * 1.06
    cx, cy = (x.max() + x.min()) / 2, (y.max() + y.min()) / 2
    return (cx - side / 2, cx + side / 2), (cy - side / 2, cy + side / 2), side


def umap_figure(title: str, xlim, ylim, side):
    """Black square panel with a boxless equal-aspect UMAP axes and short axis arrows (as in 6C)."""
    fig = plt.figure(figsize=(PANEL_MM / 25.4, PANEL_MM / 25.4), facecolor="black")
    ax = fig.add_axes([0.06, 0.06, 0.88, 0.84])
    ax.set_facecolor("black")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.text(0.06, 0.965, title, ha="left", va="top", fontsize=9, color=TEXT)

    # UMAP coordinates have no units, so two short arrows replace a box and ticks.
    length = 0.14 * side
    arrow = dict(arrowprops=dict(arrowstyle="-|>", lw=0.8, color=AXIS_ARROWS, mutation_scale=6, shrinkA=0,
                                 shrinkB=0),
                 annotation_clip=False)
    ax.annotate("", xy=(xlim[0] + length, ylim[0]), xytext=(xlim[0], ylim[0]), **arrow)
    ax.annotate("", xy=(xlim[0], ylim[0] + length), xytext=(xlim[0], ylim[0]), **arrow)
    ax.text(xlim[0] + 0.015 * side, ylim[0] - 0.012 * side, "UMAP 1", ha="left", va="top", fontsize=7,
            color=AXIS_ARROWS)
    ax.text(xlim[0] - 0.015 * side, ylim[0] + 0.015 * side, "UMAP 2", ha="right", va="bottom", rotation=90,
            fontsize=7, color=AXIS_ARROWS)
    return fig, ax


def scatter_umap(ax, cells: pd.DataFrame, colors, zorder: int, size: float = UMAP_DOT) -> None:
    ax.scatter(cells["X_umap__0"], cells["X_umap__1"], s=size, c=colors, linewidths=0, rasterized=True,
               zorder=zorder)


def key(fig, rect, entries, title=None) -> None:
    """Dot key in its own axes: the cell types top right, where 6C has its color bar; grey nuclei bottom right."""
    handles = [Line2D([], [], ls="", marker="o", ms=4, mfc=color, mec="none", label=label)
               for color, label in entries]
    kax = fig.add_axes(rect)
    kax.axis("off")
    legend = kax.legend(handles=handles, loc="upper left" if title else "lower left", fontsize=7,
                        handletextpad=0.3, labelspacing=0.4, borderaxespad=0, labelcolor=TEXT, frameon=False,
                        title=title, title_fontsize=7, alignment="left")
    if title:
        legend.get_title().set_color(TEXT)

## Run

In [ ]:
repo = find_repo(REPO)
notebook = repo / NOTEBOOK
if not notebook.is_file():
    raise FileNotFoundError(f"{notebook} not found; set NOTEBOOK to this notebook's path")

out = repo / OUT
files = {variant: [out / f"{stem}.{ext}" for ext in ("png", "svg", "pdf")] for variant, stem in VARIANTS.items()}
table_path = out / "transferred_cell_type.csv"
sidecar = out / "fig6_umap_transferred_cell_type.provenance.json"
outputs = [path for paths in files.values() for path in paths] + [table_path]
existing = [str(path) for path in outputs + [sidecar] if path.exists()]
if existing and not FORCE:
    raise FileExistsError(f"Refusing to overwrite {existing}; set FORCE = True")
out.mkdir(parents=True, exist_ok=True)

In [ ]:
joint = pd.read_parquet(repo / JOINT)
mouse_cells = pd.read_csv(repo / MOUSE_CELLS, index_col="cell")
human_cells = pd.read_csv(repo / HUMAN_CELLS, index_col="cell")
colors, labels = manuscript_colors(repo)

transfer = transfer_cell_type(repo / GRAPH_DIR, mouse_cells["cell_type"].astype(str))
human = joint[joint.species == "hs"].join(transfer, how="inner")
if len(human) != (joint.species == "hs").sum():
    raise ValueError("human nuclei of the joint export are missing from the graph block")
human["native_cell_type"] = human_cells.loc[human.index, "cell_type"].astype(str)
mouse = joint[joint.species == "mm"]

mapped = human.transferred_cell_type.notna()
unknown = set(human.loc[mapped, "transferred_cell_type"]) - set(colors)
if unknown:
    raise ValueError(f"transferred cell types without a color: {sorted(unknown)}")

# Agreement with the native human labels, over the nuclei that received a label and over all human nuclei.
agree = mapped & (human.transferred_cell_type == human.native_cell_type)
by_native = {
    native: {"agree": int(agree[group.index].sum()), "labelled": int(mapped[group.index].sum()), "all": len(group)}
    for native, group in human.groupby("native_cell_type")
}
counts = {
    "umap_human_labelled": int(mapped.sum()),
    "umap_human_no_mouse_neighbor": int((~mapped).sum()),
    "umap_mouse_context": len(mouse),
    "transferred": {str(k): int(v) for k, v in human.transferred_cell_type.value_counts().sort_index().items()},
    "exact_ties": int(human.tie.sum()),
}
agreement = {
    "agree": int(agree.sum()),
    "percent_of_labelled": round(100 * agree.sum() / mapped.sum(), 2),
    "percent_of_all_human": round(100 * agree.sum() / len(human), 2),
    "by_native_cell_type": by_native,
    "median_vote_share": round(float(human.vote_share.median()), 4),
}
LOG.info("Nuclei: %s", counts)
LOG.info("Agreement with native labels: %s", agreement)

In [ ]:
font = set_font()
LOG.info("Font: %s", font)

# Every frame shares one square fitted to the whole joint embedding, as in 6A-6C.
xlim, ylim, side = square_limits(joint)
colored = human[mapped].sample(frac=1, random_state=ORDER_SEED)
type_entries = [(colors[t], labels[t]) for t in ("endometrial_fibroblast", "decidual_cell")]
no_label = f"human, no mouse\nneighbor ({counts['umap_human_no_mouse_neighbor']})"
grey_entries = {
    "with_mouse": [(MOUSE_CONTEXT, "mouse cells"), (NO_VALUE, no_label)],
    "human_only": [(NO_VALUE, no_label)],
}

for variant in VARIANTS:
    fig, ax = umap_figure(TITLE, xlim, ylim, side)
    if variant == "with_mouse":
        scatter_umap(ax, mouse, MOUSE_CONTEXT, zorder=1)

    # Human nuclei without a label sit behind the colored ones, above the mouse context. Both cell types go down
    # in one shuffled layer, so neither hides the other where they overlap.
    scatter_umap(ax, human[~mapped], NO_VALUE, zorder=2, size=UMAP_DOT * NO_VALUE_DOT_SCALE)
    scatter_umap(ax, colored, colored.transferred_cell_type.map(colors).to_list(), zorder=3)
    key(fig, [0.80, 0.66, 0.20, 0.22], type_entries, title="transferred\nfrom mouse")
    key(fig, [0.56, 0.0, 0.44, 0.16], grey_entries[variant])
    for path in files[variant]:
        fig.savefig(path, dpi=DPI, facecolor="black")
    plt.close(fig)
    LOG.info("Wrote %s", [str(path) for path in files[variant]])

table = human[["native_cell_type", "transferred_cell_type", "vote_share", "cross_species_weight"]]
table.to_csv(table_path)
display(Image(filename=str(files["with_mouse"][0]), width=420))

In [ ]:
parameters = {"REPO": str(repo), "JOINT": JOINT, "MOUSE_CELLS": MOUSE_CELLS, "HUMAN_CELLS": HUMAN_CELLS,
              "GRAPH_DIR": GRAPH_DIR, "OUT": OUT, "FORCE": FORCE}
record = {
    "outputs": [str(path.resolve()) for path in outputs],
    "written": datetime.datetime.now().isoformat(timespec="seconds"),
    "host": platform.node(),
    "notebook": str(notebook.resolve()),
    "notebook_md5": hashlib.md5(notebook.read_bytes()).hexdigest(),
    "git": git_revision(notebook),
    "parameters": parameters,
    "inputs": [file_record(repo / JOINT, "joint"), file_record(repo / MOUSE_CELLS, "mouse_cells"),
               file_record(repo / HUMAN_CELLS, "human_cells"),
               file_record(repo / GRAPH_DIR / "C_hs_mm.npz", "graph_block")],
    "font": font,
    "versions": {"python": platform.python_version(), "matplotlib": mpl.__version__, "pandas": pd.__version__},
    "operator": "argmax over mouse cell_type of the summed SAMap connectivities to mouse neighbors "
                "(samap_label_stain_full.py vote); all mouse nuclei vote",
    "colors": {labels[t]: colors[t] for t in colors},
    "umap": counts | {"order_seed": ORDER_SEED, "panel_mm": PANEL_MM, "dpi": DPI},
    "agreement_with_native": agreement,
}
sidecar.write_text(json.dumps(record, indent=1, default=str))
LOG.info("Wrote %s", sidecar)